# Adapter Design Pattern

#### Fitting a Square Peg into a Round Hole.

The Scenario
- **The System (Target)**: A RoundHole that only accepts RoundPegs.
- **The Problem (Adaptee)**: We have a SquarePeg. It doesn't fit directly because the system checks for radius, but the square peg only has width.
- **The Solution (Adapter)**: An adapter that takes the square peg and calculates the equivalent "minimum radius" needed to fit it.

#### THE TARGET SYSTEM (Round Hole)

In [1]:
class RoundHole:
    def __init__(self, radius):
        self.radius = radius

    def fits(self, peg) -> bool:
        # The hole expects a peg with a .get_radius() method
        return self.radius >= peg.get_radius()

#### THE COMPATIBLE ITEM (Round Peg)

In [2]:
class RoundPeg:
    def __init__(self, radius):
        self.radius = radius

    def get_radius(self):
        return self.radius

#### THE INCOMPATIBLE ITEM (Square Peg)

In [3]:
class SquarePeg:
    def __init__(self, width):
        self.width = width

    def get_width(self):
        return self.width
        
    # NOTICE: This class has NO get_radius() method. 
    # It cannot be used with RoundHole directly.

#### THE ADAPTER

In [4]:
import math

class SquarePegAdapter:
    """
    This adapter wraps the SquarePeg and pretends to be a RoundPeg.
    """
    def __init__(self, peg: SquarePeg):
        self.peg = peg

    def get_radius(self):
        # Math: The minimum radius needed for a square to fit in a circle
        # is half the square's diagonal.
        # Diagonal = width * sqrt(2)
        # Radius = Diagonal / 2
        return (self.peg.get_width() * math.sqrt(2)) / 2

#### CLIENT CODE

def main():
    # A hole with radius 5
    hole = RoundHole(5)
    
    # 1. Standard Round Peg (Radius 5) -> FITS
    rpeg = RoundPeg(5)
    print(f"Round Peg (r=5) fits?  {hole.fits(rpeg)}")

    # 2. Square Peg (Width 5)
    small_sq_peg = SquarePeg(5)
    large_sq_peg = SquarePeg(10)

    # hole.fits(small_sq_peg) 
    # ^ CRASH! AttributeError: 'SquarePeg' object has no attribute 'get_radius'

    # 3. Use the Adapter!
    adapter_small = SquarePegAdapter(small_sq_peg)
    adapter_large = SquarePegAdapter(large_sq_peg)

    print(f"Square Peg (w=5) fits? {hole.fits(adapter_small)}") # True
    print(f"Square Peg (w=10) fits? {hole.fits(adapter_large)}") # False (Too big)

if __name__ == "__main__":
    main()

# Adapter Design Pattern 

explained using the Stock Market Analytics (XML vs JSON) example.

#### The Concept

The Adapter Pattern allows objects with incompatible interfaces to collaborate. It acts as a wrapper between two objects. **Analogy**: A Travel Power Adapter.
- **Client**: Your Laptop (Expects US Plug).
- **Service**: The Wall Socket in Europe (Provides EU Plug).
- **Adapter**: The device in between that translates the shape of the pins.

#### In software:

- Client: Your new App (Expects JSON).
- Service: Legacy 3rd party API (Provides XML).
- Adapter: Converts XML to JSON on the fly.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we explicitly define a "Target Interface" that the Client expects (`IAnalyticsLib`). The Adapter must implement this interface and wrap the legacy class (`LegacyStockXML`).

#### THE TARGET INTERFACE (What the App expects)

In [9]:
from abc import ABC, abstractmethod

class IAnalyticsLib(ABC):
    @abstractmethod
    def display_chart(self, json_data: str):
        pass

#### THE ADAPTEE (Incompatible Legacy Service)

In [10]:
class LegacyStockXML:
    """
    This is a 3rd party library. We cannot change its code.
    It returns data in XML format.
    """
    def get_stock_xml(self) -> str:
        return "<stock><price>150.00</price><symbol>AAPL</symbol></stock>"

#### THE ADAPTER (The Bridge)

In [12]:
class XmlToJsonAdapter(IAnalyticsLib):
    def __init__(self, xml_service: LegacyStockXML):
        self.service = xml_service

    def display_chart(self, json_data: str = ""):
        # 1. Get the data from the incompatible service
        xml_data = self.service.get_stock_xml()
        
        # 2. Convert it (Adapt logic)
        print(f"🔌 Adapter: Converting XML '{xml_data}'...")
        # (Mocking conversion logic for brevity)
        converted_json = '{"symbol": "AAPL", "price": 150.00}'
        
        # 3. Use standard logic
        print(f"📊 Displaying JSON Chart: {converted_json}")

#### CLIENT CODE

In [13]:
class StockApp:
    def main(self, lib: IAnalyticsLib):
        # The app only knows how to talk to IAnalyticsLib
        lib.display_chart("{}")

if __name__ == "__main__":
    # Setup
    old_system = LegacyStockXML()
    
    # We wrap the old system in the adapter
    adapter = XmlToJsonAdapter(old_system)
    
    # The app works happily
    app = StockApp()
    app.main(adapter)

🔌 Adapter: Converting XML '<stock><price>150.00</price><symbol>AAPL</symbol></stock>'...
📊 Displaying JSON Chart: {"symbol": "AAPL", "price": 150.00}


## The Pythonic Way

In Python, we don't strictly need Interfaces (`IAnalyticsLib`) because of **Duck Typing**. If it walks like a duck (has a `display_chart` method), it is a duck. We can also use **Class Inheritance** (Multiple Inheritance) or simply **Composition without Interfaces** to create a cleaner adapter.

A very Pythonic approach is to use `__getattr__` to delegate all other calls to the wrapped object, only intercepting the specific method we need to adapt. This makes the adapter transparent.

#### THE ADAPTEE (Legacy Service)

In [14]:
class LegacyStockXML:
    def fetch_xml(self):
        return "<stock><price>999.00</price><symbol>GOOGL</symbol></stock>"
    
    def get_status(self):
        return "Server Online"

#### THE PYTHONIC ADAPTER

In [15]:
class StockAdapter:
    def __init__(self, adaptee):
        self.adaptee = adaptee

    # The method the client expects
    def display_chart(self):
        # 1. Call legacy
        xml = self.adaptee.fetch_xml()
        
        # 2. Convert
        # Quick-and-dirty mock parsing
        symbol = xml.split("<symbol>")[1].split("</symbol>")[0]
        price = xml.split("<price>")[1].split("</price>")[0]
        json_data = {"symbol": symbol, "price": price}
        
        print(f"📊 Pythonic Chart: {json_data}")

    # MAGIC METHOD: __getattr__
    # If the client calls a method that doesn't exist here (like 'get_status'),
    # automatically delegate it to the wrapped object!
    def __getattr__(self, attr):
        return getattr(self.adaptee, attr)

#### CLIENT CODE

In [16]:
def client_code(chart_provider):
    """
    This function expects an object with .display_chart()
    """
    chart_provider.display_chart()
    
    # It might also call other methods
    if hasattr(chart_provider, 'get_status'):
        print(f"   Status: {chart_provider.get_status()}")

if __name__ == "__main__":
    print("--- Pythonic Adapter ---")
    
    legacy_api = LegacyStockXML()
    
    # Wrap it
    adapter = StockAdapter(legacy_api)
    
    # Run
    client_code(adapter)

--- Pythonic Adapter ---
📊 Pythonic Chart: {'symbol': 'GOOGL', 'price': '999.00'}
   Status: Server Online


#### Key Differences

| Feature            | Classic OOP                                                     | Pythonic                                                         |
|--------------------|-----------------------------------------------------------------|------------------------------------------------------------------|
| **Interface**      | Required (`IAnalyticsLib`).                                     | Not required (duck typing).                                     |
| **Delegation**     | Must manually implement every exposed method.                   | `__getattr__` can automatically delegate unchanged methods.     |
| **Type Checking**  | Strict (`isinstance(obj, Interface)`).                          | Loose (`hasattr(obj, "method")`).                               |


#### When to use which?

- **Java Way**: Use this in Python only if you are using type checkers like `mypy` strictly and need to enforce that the Adapter fulfills a specific abstract base class contract.
- **Pythonic Way**: Use this for most cases. The `__getattr__` trick is powerful because it allows the Adapter to be a "Transparent Proxy" that fixes the one broken method while letting all other valid methods pass through untouched.

# Here is a clear, practical example of the Adapter Design Pattern in Python.

#### The Concept
The Adapter Pattern acts as a bridge between two incompatible interfaces. It allows classes to work together that couldn't otherwise because of incompatible method names or data formats.

#### Real-world Analogy
Traveling from the US to Europe. Your US laptop plug (Client) doesn't fit the European wall socket (Service). You need a Power Adapter to sit in the middle and translate the connection.

#### The Scenario: JSON vs XML
Imagine your modern Python application expects data in JSON format. However, you need to use a legacy 3rd-party Analytics Library that only outputs XML.

Instead of rewriting the entire legacy library (which might be impossible) or changing your entire app to support XML, you build an Adapter.

#### THE TARGET (What our App expects)

In [5]:
from typing import Protocol

class ModernAnalytics(Protocol):
    """
    Our application is built to use this interface.
    It expects a simple dictionary (JSON-like) structure.
    """
    def analyze_data(self, data_json: str) -> None:
        ...

#### THE ADAPTEE (The Incompatible Legacy Class)

In [6]:
import xml.etree.ElementTree as ET

class LegacyXMLAnalytics:
    """
    This is the old 3rd-party library we MUST use.
    Problem: It only accepts XML strings, not JSON.
    """
    def analyze_xml_data(self, xml_data: str) -> None:
        # Simulate complex processing
        root = ET.fromstring(xml_data)
        print(f"Legacy Lib: Analyzing XML -> User: {root.find('user').text}, "
              f"Event: {root.find('event').text}")

#### THE ADAPTER

In [7]:
import json

class XMLAdapter:
    """
    The Adapter makes the Legacy class look like a Modern class.
    """
    def __init__(self, legacy_service: LegacyXMLAnalytics):
        self.legacy_service = legacy_service

    def analyze_data(self, data_json: str) -> None:
        """
        The key translation happens here:
        1. Receive JSON (from Client)
        2. Convert JSON to XML (for Adaptee)
        3. Call the Adaptee
        """
        print("Adapter: Converting JSON to XML...")
        
        # 1. Parse JSON
        data_dict = json.loads(data_json)
        
        # 2. Build XML string
        xml_data = (f"<root>"
                    f"<user>{data_dict['username']}</user>"
                    f"<event>{data_dict['action']}</event>"
                    f"</root>")
        
        # 3. Delegate to the legacy service
        self.legacy_service.analyze_xml_data(xml_data)

#### CLIENT CODE

In [8]:
def client_code(analytics_system: ModernAnalytics):
    """
    The client code works with any class that follows the ModernAnalytics protocol.
    It doesn't know (or care) that it's actually talking to an XML library underneath.
    """
    sample_json = '{"username": "john_doe", "action": "clicked_button"}'
    analytics_system.analyze_data(sample_json)

def main():
    # 1. We have a Legacy Service
    old_system = LegacyXMLAnalytics()
    
    # 2. We wrap it in an Adapter
    adapter = XMLAdapter(old_system)
    
    # 3. We use it as if it were a modern system
    print("--- Client: Sending JSON Data ---")
    client_code(adapter)

if __name__ == "__main__":
    main()

--- Client: Sending JSON Data ---
Adapter: Converting JSON to XML...
Legacy Lib: Analyzing XML -> User: john_doe, Event: clicked_button


# Adapter Design Pattern 

explained using a complex, real-world scenario: Unified Payment Gateway Integration.

#### The Scenario: Multi-Provider Payment System

You are building an E-Commerce platform. You need to accept payments via **Stripe** and **PayPal**.
- **The Problem**: These libraries have completely different APIs and data formats.
    - **Stripe**: Uses `cents` (Integer), requires an `auth_token`, and the method is called `create_charge`.
    - **PayPal**: Uses `dollars` (Float), requires an `api_key`, and the method is called `send_payment`.
- **The Goal**: Your checkout system should just call `process_payment(amount)` and not care which provider is running underneath.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we define a standard `IPaymentProcessor` interface. We then create explicit Adapter classes for every provider to map their specific methods and data types to our standard interface.

#### THE TARGET INTERFACE (Standard API)

In [21]:
from abc import ABC, abstractmethod

class IPaymentProcessor(ABC):
    @abstractmethod
    def pay(self, amount_in_dollars: float) -> bool:
        pass

#### THE ADAPTEES (3rd Party Libraries)

In [22]:
class StripeLibrary:
    def create_charge(self, amount_cents: int, token: str) -> str:
        print(f"💳 [Stripe] Charged {amount_cents} cents (Token: {token})")
        return "stripe_txn_123"

class PayPalLibrary:
    def send_payment(self, amount_dollars: float, user_email: str) -> str:
        print(f"🅿️ [PayPal] Sent ${amount_dollars} to {user_email}")
        return "paypal_txn_abc"

#### THE ADAPTERS

In [25]:
class StripeAdapter(IPaymentProcessor):
    def __init__(self, stripe: StripeLibrary, user_token: str):
        self.stripe = stripe
        self.user_token = user_token

    def pay(self, amount_in_dollars: float) -> bool:
        # DATA CONVERSION LOGIC: Dollars -> Cents
        amount_cents = int(amount_in_dollars * 100)
        
        # METHOD MAPPING: pay -> create_charge
        txn_id = self.stripe.create_charge(amount_cents, self.user_token)
        return txn_id is not None

class PayPalAdapter(IPaymentProcessor):
    def __init__(self, paypal: PayPalLibrary, merchant_email: str):
        self.paypal = paypal
        self.merchant_email = merchant_email

    def pay(self, amount_in_dollars: float) -> bool:
        # METHOD MAPPING: pay -> send_payment
        txn_id = self.paypal.send_payment(amount_in_dollars, self.merchant_email)
        return txn_id is not None

#### CLIENT CODE

In [26]:
def process_checkout(processor: IPaymentProcessor, amount: float):
    print("--- Starting Checkout ---")
    if processor.pay(amount):
        print("✅ Payment Successful")
    else:
        print("❌ Payment Failed")

def main():
    # Setup legacy libs
    stripe = StripeLibrary()
    paypal = PayPalLibrary()

    # Wrap them
    processor_a = StripeAdapter(stripe, "user_token_x")
    processor_b = PayPalAdapter(paypal, "shop@merchant.com")

    # The client treats them exactly the same
    process_checkout(processor_a, 50.00)  # Calls Stripe (converts to 5000 cents)
    print()
    process_checkout(processor_b, 50.00)  # Calls PayPal

if __name__ == "__main__":
    main()

--- Starting Checkout ---
💳 [Stripe] Charged 5000 cents (Token: user_token_x)
✅ Payment Successful

--- Starting Checkout ---
🅿️ [PayPal] Sent $50.0 to shop@merchant.com
✅ Payment Successful


## The Pythonic Way (Functional / Callable Wrapper)

In Python, we can simplify this drastically. Instead of creating rigid Adapter classes for every new provider, we can use a **Generic Adapter** that accepts a **Callable** (function) or simply rely on **Duck Typing**.

Here, I will show a powerful pattern: **The Mapping Adapter**. Instead of writing code to call functions, we configure the adapter with a lambda or partial function that handles the conversion logic inline. This removes the need for `StripeAdapter` and `PayPalAdapter` classes entirely.

#### THE ADAPTEES (Same 3rd Party Libs)

In [27]:
class Stripe:
    def charge(self, cents: int):
        print(f"💳 [Stripe] Processing {cents} cents")
        return True

class PayPal:
    def transfer(self, amt: float):
        print(f"🅿️ [PayPal] Processing ${amt}")
        return True

#### THE PYTHONIC ADAPTER (Generic)

In [28]:
from typing import Callable, Any

class PaymentAdapter:
    """
    A generic adapter that takes a function (the logic) rather than an object.
    It doesn't care if it's Stripe or PayPal.
    It just knows: "When I call pay(), I execute the function you gave me."
    """
    def __init__(self, pay_method: Callable[[float], Any]):
        self.pay_method = pay_method

    def pay(self, amount_dollars: float):
        # Delegate to the injected function logic
        return self.pay_method(amount_dollars)

#### CLIENT CODE

In [29]:
def checkout(adapter, amount):
    adapter.pay(amount)

def main():
    print("--- Pythonic Functional Adapter ---")
    
    stripe_api = Stripe()
    paypal_api = PayPal()

    # 1. ADAPTING STRIPE
    # We define the "Glue Logic" right here using a Lambda.
    # Logic: Convert dollars to cents, then call stripe.charge
    stripe_adapter = PaymentAdapter(
        pay_method=lambda dollars: stripe_api.charge(int(dollars * 100))
    )

    # 2. ADAPTING PAYPAL
    # Logic: Just pass dollars directly
    paypal_adapter = PaymentAdapter(
        pay_method=paypal_api.transfer
    )

    # 3. Execution
    checkout(stripe_adapter, 45.50) # Prints: 4550 cents
    checkout(paypal_adapter, 45.50) # Prints: $45.5

if __name__ == "__main__":
    main()

--- Pythonic Functional Adapter ---
💳 [Stripe] Processing 4550 cents
🅿️ [PayPal] Processing $45.5


#### Key Differences

| Feature             | Classic OOP                                                        | Pythonic                                                     |
|---------------------|--------------------------------------------------------------------|----------------------------------------------------------------|
| **Structure**       | One class per provider (`StripeAdapter`, `PayPalAdapter`).         | Single generic class or simple functions.                      |
| **Logic Location**  | Conversion logic (e.g., dollars → cents) hidden inside adapter class. | Conversion logic passed as a lambda/function during configuration. |
| **Boilerplate**     | High — new provider requires a new adapter class file.             | Low — new provider added with a single line in setup.          |


#### When to use which?

- **Java Way**: When the conversion logic is very heavy (e.g., requires 5 steps of authentication, logging, and error handling). You don't want that logic cluttering your main file.
- **Pythonic Way**: When the adaptation is simple (e.g., parameter renaming or simple unit conversion). Passing a lambda is much cleaner than writing a whole class.